In [1]:
import torch 
from torch import nn 
from nn_block import softmax 
from torch.nn import functional as F

In [2]:
def cross_entropy(inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    inputs_max = torch.max(inputs, dim=-1, keepdim=True)[0]
    inputs_shift = inputs - inputs_max 
    inputs_norm = torch.exp(inputs_shift).sum(dim = -1, keepdim=True) 
    log_softmax = inputs_shift - torch.log(inputs_norm) 
    output = - 1 * log_softmax[torch.arange(inputs.shape[0]), targets] 
    return torch.mean(output)

In [3]:
inputs = torch.randn([3, 5]) 
targets = torch.LongTensor([0, 1, 4])
print(f" my corss entropy is {cross_entropy(inputs, targets)}")
F.cross_entropy(inputs, targets)

 my corss entropy is 1.2494640350341797


tensor(1.2495)

In [5]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math
class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)
    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"]  # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]  # Get state associated with p.
                t = state.get("t", 0)  # Get iteration number from the state, or 0.
                grad = p.grad.data  # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad  # Update weight tensor in-place.
                state["t"] = t + 1  # Increment iteration number.
        return loss

In [6]:
class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-9, weight_decay=1e-4):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr, "betas": betas, 'eps':eps, 'weight_decay':weight_decay}
        super().__init__(params, defaults)
    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"]  # Get the learning rate.
            beta1, beta2 = group["betas"]
            eps = group["eps"]
            lmd = group["weight_decay"]
            for p in group["params"]:
                if p.grad is None:
                    continue 
                grad = p.grad.data 
                theta = p.data
                state = self.state[p]  # Get state associated with p.
                if len(state) == 0:
                    state['m'] = torch.zeros_like(grad)
                    state['v'] = torch.zeros_like(grad) 
                m, v = state['m'], state['v']
                t = state.get("t", 1)  # Get iteration number from the state, or 0.
                lr_t = lr * ((1 - beta2 ** t) ** 0.5 / (1 - beta1 ** t)) 
                theta.add_(theta, alpha = -lr * lmd) # 正则项修正,注意这里的参数alpha
                m.mul_(beta1).add_((1 - beta1) * grad)
                v.mul_(beta2).add_((1 - beta2) * (grad ** 2))
                theta.add_(-lr_t * m / (v ** 0.5 + eps))
                state["t"] = t + 1  # Increment iteration number.
        return loss

In [8]:
def cosine_annealing_schedule(t: int,
                              t_w: int,
                              t_c: int,
                              a_min: float,
                              a_max: float):
    if t < t_w:
        return  t / t_w * a_max 
    if t >= t_w and t < t_c:
        return a_min + 0.5 * ( 1 + math.cos((t - t_w) / (t_c - t_w) * math.pi)) * (a_max - a_min) 
    if t >= t_c:
        return a_min

In [9]:
def l2_norm_clip(parameters, max_l2_norm, epslion=1e-6):
    all_parameters = torch.cat([param.grad.flatten() for param in parameters]) 
    l2_norm = torch.sqrt(torch.sum(all_parameters ** 2)).data 
    if l2_norm >= max_l2_norm:
        for param in parameters:
            param.grad.data.mul_(max_l2_norm / (l2_norm + epslion))

In [12]:
import numpy as np
import random

In [ ]:
def data_load(x: np.ndarray, batch_size: int, context_length: int, device: str) -> tuple[torch.Tensor, torch.Tensor]:
    data_len = x.shape[0]
    choice_end = data_len - context_length - 1
    in_start = [random.randint(0, choice_end) for i in range(batch_size)] 
    nt_start = [i + 1 for i in in_start]
    in_batch = torch.stack([torch.Tensor(x[start : start + context_length]) for start in in_start], dim = 0) 
    nt_batch = torch.stack([torch.Tensor(x[start : start + context_length]) for start in nt_start], dim = 0)
    device = torch.device(device)
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    return in_batch.to(device), nt_batch.to(device)











In [29]:
data_load(np.array([i for i in range(101)]), 3, 10, torch.device('cpu'))

(tensor([[23., 24., 25., 26., 27., 28., 29., 30., 31., 32.],
         [82., 83., 84., 85., 86., 87., 88., 89., 90., 91.],
         [73., 74., 75., 76., 77., 78., 79., 80., 81., 82.]]),
 tensor([[24., 25., 26., 27., 28., 29., 30., 31., 32., 33.],
         [83., 84., 85., 86., 87., 88., 89., 90., 91., 92.],
         [74., 75., 76., 77., 78., 79., 80., 81., 82., 83.]]))

In [21]:
if torch.cuda.is_available():
    print(f"GPU 数量: {torch.cuda.device_count()}")
    print(f"当前 GPU: {torch.cuda.current_device()}")
    print(f"当前 GPU 名称: {torch.cuda.get_device_name(0)}")
    print()
    
    # 遍历所有 GPU
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        props = torch.cuda.get_device_properties(i)
        print(f"  显存: {props.total_memory / 1024**3:.2f} GB")
        print(f"  计算能力: {props.major}.{props.minor}")
        print(f"  多处理器数量: {props.multi_processor_count}")
else:
    print("没有可用的 GPU")

GPU 数量: 1
当前 GPU: 0
当前 GPU 名称: NVIDIA GeForce RTX 3080 Ti

GPU 0: NVIDIA GeForce RTX 3080 Ti
  显存: 11.63 GB
  计算能力: 8.6
  多处理器数量: 80
